# TP: Tic-Tac-Toe with Minimax and Alpha-Beta
**Fundamentals of AI**

### How to work
- Run the cells **one at a time**, from top to bottom (`Shift + Enter`).
- Read the comments: every line of code is explained.
- Answer the **questions** in the text cells (double-click on a text cell to edit it).

### Conventions used in this TP
- `'X'` is **MAX**: it wants the score to be **+1**
- `'O'` is **MIN**: it wants the score to be **-1**
- Score: **+1** = X wins, **-1** = O wins, **0** = draw
- The board is a list of 9 cells, numbered like this:

```
 0 | 1 | 2
---+---+---
 3 | 4 | 5
---+---+---
 6 | 7 | 8
```

## Part 1: The game
Nothing to change here. Just read the code and run the cells.

In [ ]:
# The 8 lines that make a win: 3 rows, 3 columns, 2 diagonals.
WINS = [(0, 1, 2), (3, 4, 5), (6, 7, 8),   # rows
        (0, 3, 6), (1, 4, 7), (2, 5, 8),   # columns
        (0, 4, 8), (2, 4, 6)]              # diagonals


def print_board(board):
    """Display the board. Empty cells show their number (0 to 8)."""
    print()
    for row in range(3):                       # row = 0, 1, 2
        cells = []                             # the 3 symbols of this row
        for col in range(3):                   # col = 0, 1, 2
            i = 3 * row + col                  # position in the list (0 to 8)
            if board[i] == ' ':                # if the cell is empty...
                cells.append(str(i))           # ...show its number
            else:
                cells.append(board[i])         # ...otherwise show X or O
        print(" " + " | ".join(cells))         # print the row: " X | 1 | O"
        if row < 2:
            print("---+---+---")               # separator between rows
    print()


def winner(board):
    """Return 'X' or 'O' if someone has won, otherwise None."""
    for a, b, c in WINS:                       # look at each winning line
        # the line wins if the 3 cells are equal and not empty
        if board[a] != ' ' and board[a] == board[b] == board[c]:
            return board[a]                    # return 'X' or 'O'
    return None                                # nobody has won (yet)


def moves(board):
    """Return the list of empty cells = the legal moves."""
    result = []                                # start with an empty list
    for i in range(9):                         # check the 9 cells
        if board[i] == ' ':                    # if the cell is empty...
            result.append(i)                   # ...it is a legal move
    return result

Let's check that these functions work. Run the next cell and look at the result.

In [ ]:
board = ['X', 'O', ' ',
         ' ', 'X', ' ',
         ' ', ' ', 'O']

print_board(board)
print("Winner:", winner(board))
print("Legal moves:", moves(board))

With X at cells 0, 1, and 2, `winner(board)` returns `'X'`.

## Part 2: Minimax (complete code, read it line by line)
`minimax(board, player)` returns the **value of a position** when both players play **perfectly**.
- **Step 1:** is the game over? Then return the score (this is the stopping case of the recursion).
- **Step 2:** otherwise try every legal move, ask `minimax` what the resulting position is worth, and keep the **highest** score if the player is MAX, or the **lowest** if the player is MIN.

The variable `nodes` counts how many positions the algorithm visits.

In [ ]:
nodes = 0   # counts how many positions the algorithm visits


def minimax(board, player):
    """
    Return the value of the position `board` when it is `player`'s turn,
    assuming both players play PERFECTLY.
    """
    global nodes                # we want to modify the global counter
    nodes += 1                  # one more position visited

    # ---- Step 1: is the game over? (base case of the recursion) ----
    w = winner(board)           # did someone win?
    if w == 'X':
        return 1                # X won: best result for MAX
    if w == 'O':
        return -1               # O won: best result for MIN
    if len(moves(board)) == 0:  # no empty cell left and no winner
        return 0                # draw

    # ---- Step 2: the game continues, try every move ----
    if player == 'X':           # X is MAX: he wants the HIGHEST score
        best = -2               # worse than any real score (-1, 0, 1)
        for m in moves(board):          # for each legal move m
            board[m] = 'X'              # 1) play the move
            score = minimax(board, 'O') # 2) ask: what is it worth? (O's turn)
            board[m] = ' '              # 3) UNDO the move (very important!)
            if score > best:            # 4) keep the highest score
                best = score
        return best

    else:                       # O is MIN: he wants the LOWEST score
        best = 2                # higher than any real score
        for m in moves(board):
            board[m] = 'O'              # play the move
            score = minimax(board, 'X') # X's turn now
            board[m] = ' '              # undo the move
            if score < best:            # keep the LOWEST score
                best = score
        return best


def best_move(board, player):
    """Return the best cell for `player` using minimax."""
    best_cell = None
    if player == 'X':                         # MAX: look for the highest score
        best_score = -2
        for m in moves(board):
            board[m] = 'X'                    # try the move
            score = minimax(board, 'O')       # evaluate the resulting position
            board[m] = ' '                    # undo it
            if score > best_score:            # better than what we had?
                best_score = score
                best_cell = m                 # remember this move
    else:                                     # MIN: look for the lowest score
        best_score = 2
        for m in moves(board):
            board[m] = 'O'
            score = minimax(board, 'X')
            board[m] = ' '
            if score < best_score:
                best_score = score
                best_cell = m
    return best_cell

### Test 1: a board where the answer is obvious
X has cells 0 and 1 and it is X's turn. Run the next cell.

In [ ]:
test = ['X', 'X', ' ',
        'O', 'O', ' ',
        ' ', ' ', ' ']

print_board(test)
nodes = 0
print("Best move for X:", best_move(test, 'X'), "| positions visited:", nodes)

**Answer:** X plays 2 because it completes the top row (0, 1, 2) and wins immediately. Playing 5 does not win.

**Answer:** `best_move(test, 'O')` returns cell 2. This blocks X's immediate win. Cell 5 is also an immediate winning move for O, so both moves have the same best minimax score (-1). The algorithm checks cell 2 first and keeps it when it later finds an equal score at cell 5.


### Test 2: the empty board
Be patient, it takes a few seconds.

In [ ]:
empty = [' '] * 9           # a list of 9 empty cells
nodes = 0
value = minimax(empty, 'X')
print("Value of the empty board:", value, "| positions visited:", nodes)

**Answer:** 0 means that with perfect play from both players, the game ends in a draw. Neither X nor O can force a win.

**Answer:** 549,946 positions are visited. The algorithm explores many possible sequences of moves, not just the 9 cells. Different move orders and game endings create many possible game states and paths.

## Part 3: Play against the AI
The function `play` lets you play against the AI and shows the board after every move.
- `human`: the symbol **you** play (`'X'` or `'O'`)
- `first`: the symbol that plays **first** (`'X'` or `'O'`)

Run the cell below once to define the function, then run the experiments.

In [ ]:
def play(human, first, ai_function=best_move):
    """
    human : 'X' or 'O'   - the symbol YOU play
    first : 'X' or 'O'   - the symbol that plays FIRST
    """
    board = [' '] * 9                  # empty board
    current = first                    # whose turn is it?

    while True:
        print_board(board)             # display the game
        w = winner(board)
        if w is not None:              # someone won: stop the game
            print("The winner is", w)
            return
        if len(moves(board)) == 0:     # board full: draw
            print("Draw!")
            return

        if current == human:           # --- human turn ---
            m = int(input("Your move (0-8): "))
            while m not in moves(board):          # check the move is legal
                m = int(input("Illegal move. Try again (0-8): "))
        else:                          # --- AI turn ---
            m = ai_function(board, current)
            print("The AI plays", m)

        board[m] = current             # put the symbol on the board
        current = 'O' if current == 'X' else 'X'   # switch the player

### Experiment A: you are X and you play first
Type a number from 0 to 8 and press Enter.

In [ ]:
play(human='X', first='X')

### Experiment B: you are X but the AI plays first
Change `first` to `'O'`.

In [ ]:
play(human='X', first='O')

### Experiment C: you are O (the AI is X, so the AI is MAX)

In [ ]:
play(human='O', first='X')

**Answer:** No. Against an AI that always plays optimally, a human cannot force a win. With perfect play, the best possible result for the human is a draw. The outcome of an actual game depends on the human's moves.

**Answer:** With the given code and move order, the AI's first move is 0 each time. The empty board is identical each game, and `best_move` checks moves in the same order. When several moves have the same score, it keeps the first one because it updates only when the score is strictly better.

**Answer:** Yes. X is MAX, so its branch chooses the move with the highest score. O is MIN, so its branch chooses the move with the lowest score. Both use the same minimax evaluation, but optimize in opposite directions.

## Part 4: Alpha-Beta
This time there is **no line-by-line explanation**. Read the code carefully, run it, and answer the questions.

Hints:
- `alpha` = the best score MAX can already guarantee (starts at -2)
- `beta` = the best score MIN can already guarantee (starts at +2)
- If `alpha >= beta`, the remaining moves cannot change the result, so we stop exploring ("pruning").

In [ ]:
def alphabeta(board, player, alpha, beta):
    global nodes
    nodes += 1

    w = winner(board)
    if w == 'X':
        return 1
    if w == 'O':
        return -1
    if len(moves(board)) == 0:
        return 0

    if player == 'X':                              # MAX
        best = -2
        for m in moves(board):
            board[m] = 'X'
            score = alphabeta(board, 'O', alpha, beta)
            board[m] = ' '
            best = max(best, score)
            alpha = max(alpha, best)
            if alpha >= beta:                      # <-- the pruning line
                break
        return best
    else:                                          # MIN
        best = 2
        for m in moves(board):
            board[m] = 'O'
            score = alphabeta(board, 'X', alpha, beta)
            board[m] = ' '
            best = min(best, score)
            beta = min(beta, best)
            if alpha >= beta:                      # <-- the pruning line
                break
        return best

### Compare the two algorithms on the empty board

In [ ]:
empty = [' '] * 9

nodes = 0
v1 = minimax(empty, 'X')
n1 = nodes

nodes = 0
v2 = alphabeta(empty, 'X', -2, 2)      # we start with alpha=-2 and beta=2
n2 = nodes

print("Minimax    : value", v1, "| positions", n1)
print("Alpha-beta : value", v2, "| positions", n2)

### Questions
Answer by reading the code and running experiments.

**Answer:** Yes. Both return 0. This is expected because alpha-beta pruning skips branches that cannot affect the final decision; it does not change the minimax value.

**Answer:** Minimax visits 549,946 positions and alpha-beta visits 18,297. The ratio is `549946 / 18297 ≈ 30.06`, so minimax visits about 30 times as many positions.

**Answer:** All possible scores are between -1 and +1. We start alpha at -2 and beta at +2 so they are outside the possible score range. This gives MAX and MIN initial bounds that cannot incorrectly prune a move before searching.

**Answer:** `break` immediately stops the current loop over the remaining moves because further exploration cannot change the result. If we remove both `break` statements, the value stays the same but more positions are visited because no branches are pruned.

**Answer:** MAX updates alpha because alpha is the best score MAX can guarantee so far. MIN updates beta because beta is the best score MIN can guarantee so far. Each branch updates the bound belonging to the player making the choice.

**Answer:** Yes. With the original move order, alpha-beta visits 18,297 positions. With the center-first order `[4, 0, 2, 6, 8, 1, 3, 5, 7]`, it visits 7,275 positions. The value remains 0. Good move ordering helps alpha-beta prune more branches, so it explores fewer positions.

**Answer:** Alpha-beta does not change the minimax value or the optimal decision; it only reduces the search. It can return a different cell when several moves are equally good and the move order or tie-breaking differs, but that move is still optimal.

## Bonus
Write `best_move_ab(board, player)`, a copy of `best_move` that calls `alphabeta` instead of `minimax`. Then play against it:

```python
play(human='X', first='O', ai_function=best_move_ab)
```

In [ ]:
def best_move_ab(board, player):
    """Return the best cell using alpha-beta pruning."""
    best_cell = None

    if player == 'X':                       # X is MAX: choose the highest score
        best_score = -2
        for m in moves(board):
            board[m] = 'X'                  # try this move
            score = alphabeta(board, 'O', -2, 2)
            board[m] = ' '                  # undo the move
            if score > best_score:
                best_score = score
                best_cell = m
    else:                                   # O is MIN: choose the lowest score
        best_score = 2
        for m in moves(board):
            board[m] = 'O'                  # try this move
            score = alphabeta(board, 'X', -2, 2)
            board[m] = ' '                  # undo the move
            if score < best_score:
                best_score = score
                best_cell = m

    return best_cell


# Quick check: this should choose cell 2 and leave the board unchanged.
test_ab = ['X', 'X', ' ',
           'O', 'O', ' ',
           ' ', ' ', ' ']
print("Best move for O:", best_move_ab(test_ab, 'O'))
print("Board after search:")
print_board(test_ab)
